In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import mixed_precision
from tensorflow.keras.layers import Layer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import glob
import os
import csv
import shutil
import time
import json
import joblib

from forecast_video_style import (
    render_feature_importance_visuals,
    render_training_metric_visuals,
)


In [26]:
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

In [27]:

# Config
CONFIG = {
    "DATA_DIR": Path("TrainingData/indicators_data/processed/stocksData"),
    "FORECAST_DIR": Path("forecasts"),
    "CACHE_DIR": Path("cache"),
    "WINDOW_SIZE": 60,
    "TRAIN_VAL_FRAC": 0.8,   # first 50% = train+val, last 50% = test (forecasts/backtest)
    "VAL_FRAC_WITHIN_TRAIN": 0.2,  # of first 50%: half train, half validation → 25% / 25% / 50%
    "MC_DROPOUT_SAMPLES": 25,
    "EXCLUDED_COLS": [
        "date", "Target_1d", "Target_1w", "Target_1m", "Target_6m",
        "polit_trade_count", "polit_purchase_count", "polit_sale_count",
        "polit_exchange_count", "polit_option_count", "polit_stock_count",
        "polit_other_asset_count", "polit_distinct_senators",
        "polit_amount_max_ord", "polit_amount_sum_logmid",
    ],
    "PROB_THRESHOLD": 0.7,
}

horizons = ["1d", "1w", "1m", "6m"]
horizon_days = [1, 5, 21, 126]

In [28]:
# Ensure dirs exist
CONFIG["FORECAST_DIR"].mkdir(exist_ok=True)
CONFIG["CACHE_DIR"].mkdir(exist_ok=True)

# Globals
scaler = StandardScaler()
feature_cols = None

In [29]:
# MC Dropout
class MCDropout(Dropout):
    def call(self, inputs, training=None):
        return super().call(inputs, training=True)

def mc_dropout_predict(model, X, n_samples=50):
    preds = np.array([model(X, training=True).numpy() for _ in range(n_samples)])
    return preds.mean(axis=0), preds.std(axis=0), preds

In [30]:
'''
Compute required return tresholds for each horizon to be considered a buy (2 sigma above average)
We can change this later to achieve higher returns but this is a good place to start for now.
'''
def compute_horizon_thresholds(df):
    thresholds = {}
    for h in ["1d", "1w", "1m", "6m"]:
        mu = df[f"Target_{h}"].mean()
        sig = df[f"Target_{h}"].std()
        thresholds[h] = mu + 1 * sig
    return thresholds

In [31]:
# Data processing helpers 
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df["Target_1d"] = np.log(df["close"].shift(-1) / df["close"])
    df["Target_1w"] = np.log(df["close"].shift(-5) / df["close"]) #Only 5 trading days in one week
    df["Target_1m"] = np.log(df["close"].shift(-21) / df["close"]) #Only 21 trading days in one month
    df["Target_6m"] = np.log(df["close"].shift(-126) / df["close"]) #Only 126 trading days in six months
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    return df

''' 
Training with overlapping horizons presents over fitting issues. To avoid this,
we can use a max-horizon step of 126 days (6 trading months). We will create
a separate process_stock for inference when it comes time to do the prediction.
This will give us daily predictions rather than a prediction every 126 days.
'''
def process_stock(csv_path: Path, for_training=True, train_cutoff=None):
    df = pd.read_csv(csv_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    df = add_features(df)

    # Compute thresholds from training period only so ~2.5% positive in train (avoids model predicting ~0)
    if for_training and train_cutoff is not None:
        df_train = df[df["date"] < pd.Timestamp(train_cutoff)]
        if len(df_train) >= 100:
            thresholds = compute_horizon_thresholds(df_train)
        else:
            thresholds = compute_horizon_thresholds(df)
    else:
        thresholds = compute_horizon_thresholds(df)

    # Create binary classification targets
    for h in ["1d", "1w", "1m", "6m"]:
        df[f"Class_{h}"] = (df[f"Target_{h}"] > thresholds[h]).astype(int)
        
    if df.empty:
        return np.array([]), np.array([]), df, np.array([])

    global feature_cols
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0.0

    features_scaled = scaler.transform(df[feature_cols].values)
    dates = df["date"].values
    target = df[["Class_1d", "Class_1w", "Class_1m", "Class_6m"]].values if for_training else None

    X, y, y_dates = [], [], []
    window = CONFIG["WINDOW_SIZE"]
    max_horizon = max(horizon_days)  # 126 days, 6 trading months
    # Use smaller step for training to get more samples per stock (21 ≈ 1 month); inference uses 126
    step = 21 if for_training else max_horizon

    for i in range(window, len(features_scaled), step):
        X.append(features_scaled[i - window:i + 1])
        if for_training:
            y.append(target[i])
        y_dates.append(dates[i])

    return (
        np.array(X),
        np.array(y) if for_training else None,
        df,
        np.array(y_dates, dtype="datetime64[ns]"),
    )

In [32]:
def process_stock_for_inference(csv_path: Path):
    df = pd.read_csv(csv_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    df = add_features(df)
    if df.empty:
        return np.array([]), np.array([]), df, np.array([])

    global feature_cols
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0.0

    features_scaled = scaler.transform(df[feature_cols].values)
    dates = df["date"].values
    target = df[["close"]].values

    X, x_dates = [], []
    window = CONFIG["WINDOW_SIZE"]
    max_horizon = 1 

    for i in range(window, len(features_scaled), max_horizon):
        X.append(features_scaled[i - window:i + 1])
        x_dates.append(dates[i])

    return (
        np.array(X),
        df.loc[window:, "close"].values[:len(X)],
        np.array(x_dates, dtype="datetime64[ns]"),
        df
    )

In [33]:
# Cache preprocessed stock
def cache_preprocessed_stock(csv_path: Path, train_cutoff, train_val_cutoff):
    stock_name = csv_path.stem
    print(f"Rebuilding cache for: {stock_name}")

    X, y, _, y_dates = process_stock(csv_path, for_training=True, train_cutoff=train_cutoff)
    if X.size == 0:
        for split in ["train", "val", "test"]:
            np.save(CONFIG["CACHE_DIR"] / f"{stock_name}_X_{split}.npy", np.array([]))
            np.save(CONFIG["CACHE_DIR"] / f"{stock_name}_y_{split}.npy", np.array([]))
        return

    y_dates = pd.to_datetime(y_dates)
    splits = {
        "train": y_dates < np.datetime64(train_cutoff),
        "val": (y_dates >= np.datetime64(train_cutoff)) & (y_dates < np.datetime64(train_val_cutoff)),
        "test": y_dates >= np.datetime64(train_val_cutoff),
    }
    for split, mask in splits.items():
        np.save(CONFIG["CACHE_DIR"] / f"{stock_name}_X_{split}.npy", X[mask])
        np.save(CONFIG["CACHE_DIR"] / f"{stock_name}_y_{split}.npy", y[mask])
        np.save(CONFIG["CACHE_DIR"] / f"{stock_name}_y_dates_{split}.npy", y_dates[mask])

    print(
        f"Cached {stock_name}: "
        + ", ".join([f"{s}={np.sum(m)}" for s, m in splits.items()])
    )

In [34]:
# Data generator
class StockDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, csv_paths, batch_size=512, split="train", shuffle=True, use_time_weights=True, decay_factor = 0.001):
        self.csv_paths = csv_paths
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.split = split
        self.use_time_weights = use_time_weights
        self.decay_factor = decay_factor
        self.windows = []
        self.stock_names = [Path(p).stem for p in csv_paths]
        self._prepare_indices()
        self.on_epoch_end()

    def _prepare_indices(self):
        self.windows = []
        self.lengths = {}
        self.date_arrays = {}
        for stock_name in self.stock_names:
            X_path = CONFIG["CACHE_DIR"] / f"{stock_name}_X_{self.split}.npy"
            y_path = CONFIG["CACHE_DIR"] / f"{stock_name}_y_{self.split}.npy"
            date_path = CONFIG["CACHE_DIR"] / f"{stock_name}_y_dates_{self.split}.npy"
            if not X_path.exists():
                n_windows = 0
            else:
                X = np.load(X_path, mmap_mode="r")
                n_windows = len(X)
                if date_path.exists():
                    self.date_arrays[stock_name] = np.load(date_path, allow_pickle=True)
            self.lengths[stock_name] = n_windows
            for i in range(n_windows):
                self.windows.append((stock_name, i))
        self.indices = np.arange(len(self.windows))

    def __len__(self):
        return int(np.ceil(len(self.windows) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch, y_batch, weights_batch = [], [], []
        cache = {}
        for bi in batch_indices:
            stock_name, win_idx = self.windows[bi]
            if stock_name not in cache:
                X = np.load(CONFIG["CACHE_DIR"] / f"{stock_name}_X_{self.split}.npy", mmap_mode="r")
                y = np.load(CONFIG["CACHE_DIR"] / f"{stock_name}_y_{self.split}.npy", mmap_mode="r")
                cache[stock_name] = (X, y)
            X_arr, y_arr = cache[stock_name]
            X_batch.append(X_arr[win_idx])
            y_batch.append(y_arr[win_idx])

            if self.use_time_weights and stock_name in self.date_arrays:
                dates = self.date_arrays[stock_name]
                date = pd.to_datetime(dates[win_idx])
                #Give more weight to recent dates now
                date_ago = (pd.Timestamp.now() - date).days
                weights_batch.append(np.exp(-self.decay_factor * date_ago))
            else:
                weights_batch.append(1.0)

        return (np.array(X_batch, dtype=np.float32), 
                np.array(y_batch, dtype=np.float32),
                np.array(weights_batch, dtype=np.float32))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [35]:
def make_forecast(model, X, dates, closes, horizons, horizon_days):
    y_pred_mean, y_pred_std, _ = mc_dropout_predict(model, X, n_samples=CONFIG["MC_DROPOUT_SAMPLES"])
    df_dict = {"Date": dates, "Close": closes}

    for i, h in enumerate(horizons):
        df_dict[f"Pred_Prob_{h}"] = y_pred_mean[:, i]
        df_dict[f"Pred_Prob_Std_{h}"] = y_pred_std[:, i]   

    forecasting_df = pd.DataFrame(df_dict)
    return forecasting_df

all_csvs = sorted(glob.glob(str(CONFIG["DATA_DIR"] / "*.csv")))
print(f"Found {len(all_csvs)} stocks")

Found 348 stocks


In [36]:
# Determine global cutoffs & fit scaler
global_min_date, global_max_date = None, None
scaler_inputs = []
feature_cols = None

for csv_path in all_csvs:
    df_tmp = pd.read_csv(csv_path, parse_dates=["date"])
    dates_valid = df_tmp["date"].dropna()
    if len(dates_valid) == 0:
        continue
    dmin, dmax = dates_valid.min(), dates_valid.max()
    if global_min_date is None or dmin < global_min_date:
        global_min_date = dmin
    if global_max_date is None or dmax > global_max_date:
        global_max_date = dmax

train_val_cutoff = global_min_date + (global_max_date - global_min_date) * CONFIG["TRAIN_VAL_FRAC"]
train_cutoff = global_min_date + (train_val_cutoff - global_min_date) * (1 - CONFIG["VAL_FRAC_WITHIN_TRAIN"])
CONFIG["FORECAST_DIR"].mkdir(exist_ok=True)
(CONFIG["FORECAST_DIR"] / "oos_start_date.txt").write_text(str(train_val_cutoff.date()))
import json
split_info = {
    "train_start": str(global_min_date.date()),
    "train_end": str(train_cutoff.date()),
    "val_start": str(train_cutoff.date()),
    "val_end": str(train_val_cutoff.date()),
    "oos_start": str(train_val_cutoff.date()),
    "data_end": str(global_max_date.date()),
}
(CONFIG["FORECAST_DIR"] / "split_info.json").write_text(json.dumps(split_info, indent=2))
print("Training window:  ", split_info["train_start"], "to", split_info["train_end"])
print("Validation window:", split_info["val_start"], "to", split_info["val_end"])
print("OOS backtest:     ", split_info["oos_start"], "to", split_info["data_end"], "(model not trained on this)")

for csv_path in all_csvs:
    df = pd.read_csv(csv_path, parse_dates=["date"]).sort_values("date").dropna()
    df = add_features(df)
    if df.empty:
        continue
    feat_cols = [c for c in df.columns if c not in CONFIG["EXCLUDED_COLS"]]
    if feature_cols is None:
        feature_cols = feat_cols
    train_rows = df[df["date"] < train_cutoff]
    if len(train_rows) > 0:
        scaler_inputs.append(train_rows[feat_cols].values)

X = np.vstack(scaler_inputs)
X = np.nan_to_num(X, nan=0.0, posinf=1e6, neginf=-1e6)
scaler.fit(X)
print("Fitted scaler. feature_cols:", feature_cols)


Training window:   2011-03-29 to 2020-11-17
Validation window: 2020-11-17 to 2023-04-17
OOS backtest:      2023-04-17 to 2026-04-22 (model not trained on this)
Fitted scaler. feature_cols: ['close', 'YesterdayClose', 'YesterdayOpenLogR', 'YesterdayHighLogR', 'YesterdayLowLogR', 'YesterdayVolumeLogR', 'YesterdayCloseLogR', 'MA10', 'MA20', 'MA30', 'DayOfWeek', 'DayOfMonth', 'MonthNumber', 'EMA10', 'EMA30', 'RSI', 'MACD', 'MACD_Signal', 'BollingerUpper', 'BollingerLower', 'Volatility_10', 'Volatility_20', 'Volatility_30', 'OBV', 'ZScore', 'insider_shares', 'insider_amount', 'insider_buy_flag', 'sentiment', 'num_articles', 'fear_greed', 'fear_greed_correlation', 'overnight_gap', 'abnormal_vol', 'volatility_5d', 'volatility_20d', 'momentum_5d', 'momentum_20d', 'skew_5d', 'intraday_range', 'sentiment_change']


In [37]:
time.sleep(5)
# Cache all data 
if CONFIG["CACHE_DIR"].exists():
    shutil.rmtree(CONFIG["CACHE_DIR"])
CONFIG["CACHE_DIR"].mkdir(exist_ok=True)

print("Caching all preprocessed stock data...")
for csv_path in all_csvs:
    cache_preprocessed_stock(Path(csv_path), train_cutoff, train_val_cutoff)
print("Done caching.")

Caching all preprocessed stock data...
Rebuilding cache for: AACBR_daily_processed
Rebuilding cache for: AACBU_daily_processed
Rebuilding cache for: AACB_daily_processed
Rebuilding cache for: AACG_daily_processed
Cached AACG_daily_processed: train=113, val=29, test=30
Rebuilding cache for: AAL_daily_processed
Cached AAL_daily_processed: train=113, val=29, test=30
Rebuilding cache for: AA_daily_processed
Cached AA_daily_processed: train=41, val=28, test=30
Rebuilding cache for: AGNCP_daily_processed
Cached AGNCP_daily_processed: train=1, val=29, test=30
Rebuilding cache for: AGNCZ_daily_processed
Rebuilding cache for: AGO_daily_processed
Cached AGO_daily_processed: train=113, val=29, test=30
Rebuilding cache for: AGRO_daily_processed
Cached AGRO_daily_processed: train=109, val=29, test=30
Rebuilding cache for: AGRZ_daily_processed
Rebuilding cache for: AGX_daily_processed
Cached AGX_daily_processed: train=113, val=29, test=30
Rebuilding cache for: AGYS_daily_processed
Cached AGYS_daily_

In [38]:
class Attention(Layer):
    def __init__(self):
        super(Attention, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros")        
        super().build(input_shape)

    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)  # attention weights
        output = x * a
        return tf.keras.backend.sum(output, axis=1)

In [39]:
# Data generators
train_gen = StockDataGenerator(all_csvs, batch_size=128, split="train", shuffle=True, use_time_weights=False, decay_factor=0.002)
val_gen = StockDataGenerator(all_csvs, batch_size=128, split="val", shuffle=False)

In [40]:
# Why training can finish in "no time": total samples and steps per epoch
total_train_windows = len(train_gen.windows)
total_val_windows = len(val_gen.windows)
steps_per_epoch = len(train_gen)
print(f"Training windows (samples): {total_train_windows}  |  Val windows: {total_val_windows}")
print(f"Steps per epoch (train): {steps_per_epoch}  (batch_size=128 → ~{total_train_windows // 128}–{total_train_windows // 128 + 1} steps)")
print("With 50/50 split, process_stock uses one sample every 126 days → few samples per stock → few steps/epoch. EarlyStopping stops when val_loss doesn't improve for patience=25 epochs.")

Training windows (samples): 16588  |  Val windows: 6730
Steps per epoch (train): 130  (batch_size=128 → ~129–130 steps)
With 50/50 split, process_stock uses one sample every 126 days → few samples per stock → few steps/epoch. EarlyStopping stops when val_loss doesn't improve for patience=25 epochs.


In [41]:
# Actual training with the weights now
def generator_with_weights(gen):
    for X, y, w in gen:
        yield X, y, w

train_dataset = tf.data.Dataset.from_generator(
    lambda: generator_with_weights(train_gen),
    output_signature=(
        tf.TensorSpec(shape=(None, CONFIG["WINDOW_SIZE"] + 1, len(feature_cols)), dtype=tf.float32),
        tf.TensorSpec(shape=(None, 4), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32),
    )
)

val_dataset = tf.data.Dataset.from_generator(
    lambda: generator_with_weights(val_gen),
    output_signature=(
        tf.TensorSpec(shape=(None, CONFIG["WINDOW_SIZE"] + 1, len(feature_cols)), dtype=tf.float32),
        tf.TensorSpec(shape=(None, 4), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32),
    )
)

##### **AUC (Area Under the ROC Curve)**

AUC measures how well the model **ranks positive outcomes higher than negative ones**.  
In other words, if you randomly pick one example where the stock *did exceed* the target threshold  
and one where it *didn’t*, AUC tells you how often the model assigns a higher probability  
to the correct (positive) case.

This is extremely important for my forecasting code because the backtest selects trades by  
**choosing the horizon with the highest predicted probability** each day.  
The model is not judged by whether its prediction crosses a threshold like 0.5 -  
it is judged by how well it *ranks* stronger opportunities above weaker ones.  

**AUC scale:**

- **0.5** → Random guessing (no predictive power)  
- **0.6** → Slightly better than random (weak but may still be usable in trading)  
- **0.8** → Good (model reliably separates strong opportunities from weak ones)  
- **1.0** → Perfect separation (unrealistic for financial markets)

Higher AUC means the model is better at identifying which future returns are likely  
to exceed the "(mean + 2sigma)" threshold - which directly improves the decision-making in my backtest.


In [42]:
from tensorflow.keras.metrics import AUC
n_features = len(feature_cols)
model = Sequential([
    Conv1D(32, kernel_size=3, activation="relu", 
           input_shape=(CONFIG["WINDOW_SIZE"] + 1, n_features)),
    BatchNormalization(),
    MCDropout(0.3),
    LSTM(64, return_sequences=False), 
    MCDropout(0.3),
    Dense(32, activation="relu"),
    Dense(4, activation="sigmoid") #Four time horizon predictions as output
])

# Don't consider stopping until at least 15 epochs (avoids stopping at 1/500)
class EarlyStoppingWithWarmup(EarlyStopping):
    def __init__(self, start_epoch=15, **kwargs):
        super().__init__(**kwargs)
        self.start_epoch = start_epoch
    def on_epoch_end(self, epoch, logs=None):
        if epoch >= self.start_epoch:
            super().on_epoch_end(epoch, logs)

early_stop = EarlyStoppingWithWarmup(
    start_epoch=15,
    patience=50,
    monitor="val_loss",
    restore_best_weights=True,
    mode="min",
)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=[AUC(name="auc")])
history = model.fit(train_gen, validation_data=val_gen, callbacks=[early_stop], epochs=500)
#history = model.fit(train_gen, validation_data=val_gen,epochs=2)

Path("output_plots").mkdir(exist_ok=True)

render_training_metric_visuals(
    history.history["loss"],
    history.history["val_loss"],
    "output_plots/training_loss.png",
    "output_plots/training_loss.mp4",
    title="Training Loss",
    y_label="Loss",
)
print("Saved output_plots/training_loss.png")
print("Saved output_plots/training_loss.mp4")

render_training_metric_visuals(
    history.history["auc"],
    history.history["val_auc"],
    "output_plots/training_auc.png",
    "output_plots/training_auc.mp4",
    title="Training AUC",
    y_label="AUC",
    y_max=1.0,
)
print("Saved output_plots/training_auc.png")
print("Saved output_plots/training_auc.mp4")


Epoch 1/500
130/130 [==============================] - 15s 100ms/step - loss: 0.3736 - auc: 0.5412 - val_loss: 0.0776 - val_auc: 0.5461
Epoch 2/500
130/130 [==============================] - 10s 79ms/step - loss: 0.3298 - auc: 0.5965 - val_loss: 0.0777 - val_auc: 0.5507
Epoch 3/500
130/130 [==============================] - 11s 82ms/step - loss: 0.3260 - auc: 0.6199 - val_loss: 0.0762 - val_auc: 0.5694
Epoch 4/500
130/130 [==============================] - 10s 78ms/step - loss: 0.3232 - auc: 0.6357 - val_loss: 0.0769 - val_auc: 0.5717
Epoch 5/500
130/130 [==============================] - 12s 89ms/step - loss: 0.3217 - auc: 0.6438 - val_loss: 0.0769 - val_auc: 0.5761
Epoch 6/500
130/130 [==============================] - 11s 84ms/step - loss: 0.3204 - auc: 0.6484 - val_loss: 0.0776 - val_auc: 0.5694
Epoch 7/500
130/130 [==============================] - 10s 80ms/step - loss: 0.3185 - auc: 0.6563 - val_loss: 0.0776 - val_auc: 0.5687
Epoch 8/500
130/130 [==============================] -

In [43]:
model.save(MODELS_DIR / "latest_model.keras")
joblib.dump(scaler, MODELS_DIR / "latest_scalar.pkl")

meta = {
    "feature_cols": feature_cols,
    "window_size": CONFIG["WINDOW_SIZE"],
    "horizons": ["1d","1w","1m","6m"],
}

(MODELS_DIR / "latest_meta.json").write_text(json.dumps(meta, indent=2))
print("Saved model.")

Saved model.


In [44]:
# Training metric visuals are rendered in the training cell above.


In [45]:
from sklearn.metrics import mean_squared_error

def feature_importance(model, X_val, y_val, feature_names):
    base_preds = model.predict(X_val)
    base_loss = mean_squared_error(y_val, base_preds)
    importances = []

    for i, col in enumerate(feature_names):
        X_val_permuted = X_val.copy()
        np.random.shuffle(X_val_permuted[:, :, i])
        preds = model.predict(X_val_permuted)
        loss = mean_squared_error(y_val, preds)
        importances.append(loss - base_loss)

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances,
    }).sort_values("Importance", ascending=False)

    return importance_df

In [46]:
batch = val_gen[0]
if isinstance(batch, tuple):
    if len(batch) == 2:
        X_val, y_val = batch
    elif len(batch) == 3:
        X_val, y_val, _ = batch  # ignore sample weights
    else:
        raise ValueError(f"Unexpected number of outputs: {len(batch)}")
else:
    raise ValueError("val_gen[0] did not return a tuple.")
importance_df = feature_importance(model, X_val, y_val, feature_cols)

Path("output_plots").mkdir(exist_ok=True)
importance_df.to_csv(Path("output_plots") / "feature_importance.csv", index=False)
render_feature_importance_visuals(
    importance_df,
    CONFIG["FORECAST_DIR"] / "feature_importance.png",
    Path("output_plots") / "feature_importance.mp4",
    title="Feature Importance",
)
print("Saved output_plots/feature_importance.csv")
print(f"Saved {CONFIG['FORECAST_DIR'] / 'feature_importance.png'}")
print("Saved output_plots/feature_importance.mp4")


4/4 [==============================] - 0s 8ms/step
Saved output_plots/feature_importance.csv
Saved forecasts\feature_importance.png
Saved output_plots/feature_importance.mp4


In [47]:
# Feature importance visuals are rendered in the previous cell.


In [ ]:
# Forecasting 
for file in glob.glob(str(CONFIG["FORECAST_DIR"] / "*.csv")):
    os.remove(file)

print("Generating forecasts...")

for csv_path in all_csvs:
    stock_name = Path(csv_path).stem

    X_all, closes, pred_dates, df = process_stock_for_inference(Path(csv_path))

    # Only forecast for the test period (last 20%) so validation/backtest use OOS only
    oos = np.datetime64(pd.Timestamp(train_val_cutoff))
    pred_flat = np.ravel(pred_dates)
    if pred_flat.dtype.kind == "M":
        mask = pred_flat >= oos
    else:
        # float/object: convert via list so pd.to_datetime doesn't treat array as mapping
        pred_dt = pd.to_datetime(pred_flat.tolist(), errors="coerce").values
        mask = pred_dt >= oos
    X_all, closes, pred_dates = X_all[mask], closes[mask], pred_dates[mask]

    # Skip if nothing to forecast
    if X_all.size == 0:
        continue

    # Create forecast dataframe
    forecast_df = make_forecast(
        model=model,
        X=X_all,
        dates=pred_dates,
        closes=closes,
        horizons=horizons,
        horizon_days=horizon_days
    )

    forecast_df.to_csv(CONFIG["FORECAST_DIR"] / f"{stock_name}_forecast.csv", index=False)
    print("Saved:", stock_name)

print("Done.")

Generating forecasts...
Saved: AACG_daily_processed
Saved: AAL_daily_processed
Saved: AA_daily_processed
Saved: AGNCP_daily_processed
Saved: AGO_daily_processed
Saved: AGRO_daily_processed
Saved: AGX_daily_processed
Saved: AGYS_daily_processed
Saved: AHCO_daily_processed
Saved: AHG_daily_processed
Saved: AHR_daily_processed
Saved: AHT_daily_processed
Saved: AIFF_daily_processed
Saved: AIG_daily_processed
Saved: AIHS_daily_processed
Saved: AIIOW_daily_processed
Saved: AIIO_daily_processed
Saved: AIMDW_daily_processed
Saved: AIMD_daily_processed
Saved: AIM_daily_processed
Saved: AIN_daily_processed
Saved: AIOT_daily_processed
Saved: AIO_daily_processed
Saved: AIP_daily_processed
Saved: AIRE_daily_processed
Saved: AIRG_daily_processed
Saved: AIRI_daily_processed
Saved: AIRJW_daily_processed
Saved: AIRJ_daily_processed
Saved: AIRS_daily_processed
Saved: AIRTP_daily_processed
Saved: AIRT_daily_processed
Saved: AIR_daily_processed
Saved: AISPW_daily_processed
Saved: AISP_daily_processed
Save

: 